# Automated ML Project Architect

In [31]:
# Importing Necessary Libraries:

import os
from dotenv import load_dotenv
import pandas as pd
from openai import OpenAI
from IPython.display import Markdown, display

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')
openai = OpenAI()

## Step-1: Pandas Extractor

In [2]:
data = pd.read_csv('loan_borowwer_data.csv')
data.head()

,credit.policy,purpose,int.rate,installment,log.annual.inc,dti,fico,days.with.cr.line,revol.bal,revol.util,inq.last.6mths,delinq.2yrs,pub.rec,not.fully.paid
0,1,debt_consolidation,0.1189,829.10,11.350407,19.48,737,5639.958333,28854,52.1,0,0,0,0
1,1,credit_card,0.1071,228.22,11.082143,14.29,707,2760.000000,33623,76.7,0,0,0,0
2,1,debt_consolidation,0.1357,366.86,10.373491,11.63,682,4710.000000,3511,25.6,1,0,0,0
3,1,debt_consolidation,0.1008,162.34,11.350407,8.10,712,2699.958333,33667,73.2,1,0,0,0
4,1,credit_card,0.1426,102.92,11.299732,14.97,667,4066.000000,4740,39.5,0,1,0,0


In [14]:
# Function to Load in CSV and Getting Columns, Data-Types and First Few Rows:

def data_profile(file_path):

    # Loading Given CSV File:
    df = pd.read_csv(file_path)

    # Getting Column Names:
    column_names = df.columns.to_list()
    #print(column_names)

    # Getting Data Types:
    data_types = df.dtypes.to_list()
    #print(data_types)

    # Getting First few Rows for Example Data:
    data_sample = df.head().to_string(header= False)
    #print(data_sample)

    # Getting Missing value Counts:
    missing_count = df.isnull().sum().to_string()
    #print(missing_count)

    # Getting Summary Statistics:
    summary_stats = df.describe().to_string()
    #print(summary_stats)

    return column_names, data_types, data_sample, missing_count, summary_stats

In [15]:
# Testing Above function:
a,b,c,d,e = data_profile(file_path= 'C://Users//shail//PycharmProjects//PythonProject//Applied-LLM-Engineering//01_Exploring_Top_Models//loan_borowwer_data.csv')

In [16]:
print(a, b, c, d, e)

['credit.policy', 'purpose', 'int.rate', 'installment', 'log.annual.inc', 'dti', 'fico', 'days.with.cr.line', 'revol.bal', 'revol.util', 'inq.last.6mths', 'delinq.2yrs', 'pub.rec', 'not.fully.paid'] [dtype('int64'), dtype('O'), dtype('float64'), dtype('float64'), dtype('float64'), dtype('float64'), dtype('int64'), dtype('float64'), dtype('int64'), dtype('float64'), dtype('int64'), dtype('int64'), dtype('int64'), dtype('int64')] 0  1  debt_consolidation  0.1189  829.10  11.350407  19.48  737  5639.958333  28854  52.1  0  0  0  0
1  1         credit_card  0.1071  228.22  11.082143  14.29  707  2760.000000  33623  76.7  0  0  0  0
2  1  debt_consolidation  0.1357  366.86  10.373491  11.63  682  4710.000000   3511  25.6  1  0  0  0
3  1  debt_consolidation  0.1008  162.34  11.350407   8.10  712  2699.958333  33667  73.2  1  0  0  0
4  1         credit_card  0.1426  102.92  11.299732  14.97  667  4066.000000   4740  39.5  0  1  0  0 credit.policy        0
purpose              0
int.rate    

In [32]:
# Writing a Function for API Call to LLM:

def pandas_extractor(column_names, data_types, data_sample, missing_count, summary_stats):

    # Defining System prompt:
    system_prompt = """
    You are a Senior Data Scientist. Your Job is to analyse the given Data Profile and help the user with deciding Data Cleaning Process and Problem Type and Model Recommendations.

    Output a JSON object containing the target_variable, problem_type, recommended_models, and specific data_cleaning_warnings.

    Output Should be in this Specific JSON Format:
    {
    "target_variable": "Churn",
    "problem_type": "Binary Classification",
    "recommended_models": ["Logistic Regression", "Support Vector Machine"],
    "data_cleaning_warnings": ["'Date' column is a string", "'Age' has missing values"]
    }
    """

    # Defining User prompt:
    user_prompt = f"""
    Here are the column names:
    {column_names}

    Here are the data types:
    {data_types}

    Here is a sample of the data:
    {data_sample}

    Here is the count of missing values per column:
    {missing_count}

    Here are the summary statistics:
    {summary_stats}

    Based on this complete data profile, please help me decide data cleaning steps, problem type and model recommendations.

    """

    # Defining Message:
    messages = [
        {'role': 'system', 'content': system_prompt},
        {'role': 'user', 'content': user_prompt},
    ]

    # Calling LLM API:
    response = openai.chat.completions.create(model= 'gpt-5-mini',
                                              messages= messages,
                                              response_format= {'type': 'json_object'})

    return response.choices[0].message.content


In [33]:
# Testing pandas_extractor Function:
column_names, data_types, data_sample, missing_count, summary_stats = data_profile('loan_borowwer_data.csv')

test_res = pandas_extractor(column_names= column_names,
                            data_types= data_types,
                            data_sample= data_sample,
                            missing_count= missing_count,
                            summary_stats= summary_stats)

In [34]:
print(test_res)

{
  "target_variable": "not.fully.paid",
  "problem_type": "Binary Classification",
  "recommended_models": [
    "Logistic Regression (with class_weight or regularization)",
    "Random Forest",
    "Gradient Boosting (XGBoost or LightGBM)",
    "CatBoost (handles categorical 'purpose' well)",
    "Support Vector Machine (with scaling) — for smaller experiments"
  ],
  "data_cleaning_warnings": [
    "'purpose' is object/categorical — requires encoding (one-hot, target/frequency encoding, or use CatBoost's native handling)",
    "'revol.bal' is highly skewed with a very large max (~1.2e6) — check for outliers, consider capping or log-transform",
    "'installment' and 'revol.bal' distributions are skewed — consider transformation or robust scaling for some models",
    "'log.annual.inc' is already log-transformed — do not re-log; consider exponentiating only for interpretability if needed",
    "'int.rate' is expressed as a decimal (e.g. 0.1189 = 11.89%) — ensure consistent interpreta

## Step-2: The Tech Writer

In [37]:
# Defining a Function to Write A Project Markdown File Using LLM API Call:

def tech_writer(json_summary):

    system_prompt = """
    You are to Act as an Senior Tech Writer. Your job is to analyze given JSON Summary of Data and Recommendations and Strategy by Senior Data Scientist and write A Highly Professional, Step by Step, formatted README.md file for a GitHub Repository.

    """

    user_prompt = f"""
    Here is Recommendations and Strategy Given by Senior Data Scientist: {json_summary}.
    Please create a Professional Markdown Document explaining the business problem, why those specific models were chosen, and what data cleaning steps need to happen first.
    """

    # Defining Message:
    messages = [
        {'role': 'system', 'content': system_prompt},
        {'role': 'user', 'content': user_prompt},
    ]

    # Defining Call to LLM API:
    response = openai.chat.completions.create(model= 'gpt-5-mini',
                                              messages= messages)

    return response.choices[0].message.content


In [36]:
# Testing Above Function:
column_names, data_types, data_sample, missing_count, summary_stats = data_profile('loan_borowwer_data.csv')

test_res = pandas_extractor(column_names= column_names,
                            data_types= data_types,
                            data_sample= data_sample,
                            missing_count= missing_count,
                            summary_stats= summary_stats)

markdown_doc = tech_writer(json_summary=test_res)

display(Markdown(markdown_doc))

# README — Credit Default Prediction (not.fully.paid)
This repository contains a predictive modelling workflow for the binary classification problem: predicting whether a loan applicant will NOT fully repay a loan (`not.fully.paid`). This README summarizes the business objective, recommended modelling strategy, and a prioritized, step‑by‑step data cleaning and preprocessing plan prepared by the Senior Data Scientist.

---

## Business problem (concise)
We want to predict the probability that a borrower will not fully repay a loan (`not.fully.paid`). Accurate predictions enable:
- Better credit risk decisions (approve/decline, pricing, or setting covenants),
- Targeted collections and loss mitigation strategies,
- Capital allocation and portfolio management improvements.

Key constraints and considerations:
- The target is imbalanced (~16% positive class).
- Business decisions often require well‑calibrated probabilities (not just ranking).
- Interpretability and regulatory explainability may be required for some stakeholders.

---

## Recommended modelling approach — high level
Target: `not.fully.paid` (binary classification)

Primary modeling goals:
1. Reliable discrimination between good and bad loans (AUC / PR).
2. Well‑calibrated probabilities for decision thresholds and expected loss calculations.
3. Interpretability for risk teams (where needed).
4. Robustness to noise and skewed numerical distributions.

Recommended model family mix:
- Logistic Regression (with class_weight or regularization)
- Random Forest
- Gradient Boosting (XGBoost / LightGBM / CatBoost)
- Support Vector Machine (with feature scaling)
- Calibrated probability models (Platt scaling / isotonic)

Rationale for each model:
- Logistic Regression
  - Fast, interpretable baseline.
  - Well suited for regulatory/explainability needs.
  - Use class_weight or penalized regularization to address imbalance and prevent overfitting.
- Random Forest
  - Nonlinear relationships and interactions without heavy tuning.
  - Robust to outliers and feature scaling.
  - Provides feature importance for insights.
- Gradient Boosting (XGBoost/LightGBM/CatBoost)
  - Often highest predictive performance on tabular data.
  - Handles missing values (some implementations) and complex interactions.
  - CatBoost natively handles categorical variables (optionally avoiding manual encoding).
- Support Vector Machine (SVM)
  - Useful as a complementary model for margin-based separation.
  - Works well in lower-dimensional feature sets after careful scaling.
  - Less scalable to very large datasets—use as a comparative baseline where feasible.
- Calibrated Probability Models
  - Platt scaling or isotonic regression (e.g., sklearn.CalibratedClassifierCV) to produce well-calibrated probabilities, important for expected loss and decision thresholding.

Evaluation and selection strategy:
- Use stratified cross‑validation with appropriate scoring (ROC‑AUC, PR‑AUC, Brier score, calibration plots).
- Include calibration (Brier / calibration curve) as an evaluation objective, not only discrimination.
- Tune hyperparameters via nested or repeated stratified CV to avoid information leakage.

---

## Data cleaning & preprocessing — prioritized checklist
Below is a step‑by‑step, prioritized checklist to perform before model training. Items are grouped by priority: MUST (do before any modelling), SHOULD (strongly recommended), and OPTIONAL (useful enhancements).

### MUST — immediate checks and corrections
1. Data ingestion verification
   - Confirm there are actually no missing values: check for NaN, empty strings, whitespace-only strings, and special placeholders (e.g., "NA", "-").
   - Example checks: trim whitespace in object columns, count unique values and null-like tokens.

2. Target distribution and splits
   - Confirm label balance: positive class ~16%. This is imbalanced and impacts evaluation and sampling strategies.
   - Always use stratified train/validation/test splits to preserve class proportions (e.g., sklearn.model_selection.train_test_split with `stratify=y`).

3. Categorical column(s) (notably `purpose`)
   - Inspect unique categories, spelling inconsistencies, mixed cases, and rare categories.
   - Normalize categories (lowercase, strip whitespace, correct obvious typos).
   - Decide encoding strategy:
     - One‑hot encoding for low‑cardinality categories.
     - Target/ordinal encoding or frequency encoding for many categories (use cross‑validation to avoid leakage).
     - CatBoost encoding or using CatBoost model directly to avoid manual encoding.

4. Confirm units and semantics
   - `int.rate` appears as a decimal fraction (e.g., 0.1226). Confirm whether this is fraction (12.26%) or already percent.
   - `log.annual.inc` is a logged income. Decide whether to keep log-scale (recommended for modeling) or exponentiate for interpretability.

5. Duplicate rows
   - Check for and handle duplicate rows where duplicates represent the same loan event. Remove or deduplicate as appropriate.

### SHOULD — preprocessing to improve model robustness
6. Outlier and skew handling
   - Identify extreme outliers (e.g., `revol.bal` max ≈1,207,359 while 75% ≈18k).
   - Options:
     - Winsorize or cap extreme values at a high percentile (e.g., 99th/99.9th).
     - Log-transform highly right-skewed variables (add offset for zeros).
     - Use RobustScaler for models sensitive to outliers.
   - Keep copies of original and transformed features for interpretability checks.

7. Handle suspicious values / units
   - `revol.util` >100 (max=119): inspect data entry and business logic.
     - If >100 is impossible by definition, cap at 100 or treat as missing and impute.
     - Document decisions and thresholds.

8. Scaling numeric features
   - For distance-based or regularized models (SVM, LogisticRegression with L2/L1), scale numeric features:
     - StandardScaler (mean=0, std=1) or RobustScaler (median + IQR) when outliers are present.
   - Tree-based models (Random Forest, Gradient Boosting) do not require scaling.

9. Multicollinearity and correlated features
   - Compute correlation matrix and Variance Inflation Factor (VIF) for numeric features.
   - If high multicollinearity exists (e.g., `installment` correlates with loan amount), consider:
     - Removing one of the collinear features,
     - Combining features,
     - Using dimensionality reduction (e.g., PCA) as an optional strategy.

10. Feature engineering suggestions (high impact)
   - Create interaction features where business makes sense (e.g., `int.rate * installment`, `loan_amount / log.annual.inc`).
   - Bucket continuous features that have non-linear effects (e.g., age of credit line buckets).
   - Derive ratio features (revolving utilization ratios, debt-to-income proxies).
   - Capture recency or categorical cardinality features (e.g., encoding `purpose` frequency).

11. Address label imbalance
   - Strategies to try (apply within cross‑validation or training pipeline to avoid leakage):
     - class_weight (in LogisticRegression, RandomForest, many sklearn models),
     - sample weights (XGBoost/LightGBM scale_pos_weight),
     - Resampling: SMOTE/ADASYN (oversampling) or random undersampling (use care to not lose signal),
     - Ensemble methods that are robust to imbalance (e.g., gradient boosting with proper objective).
   - Evaluate performance on both ROC-AUC and Precision-Recall (PR-AUC), especially since positive class is minority.

### OPTIONAL — additional quality checks and improvements
12. Missing value imputation strategy
   - If missing values appear after ingestion, choose imputation method by feature type:
     - Numeric: median (robust) or model-based imputation.
     - Categorical: “missing” category or most frequent value.
   - Consider iterative imputation when relationships are strong.

13. Temporal leakage checks
   - If dataset spans time, ensure features that would not be available at decision time are excluded.
   - Create a time-based train/test split if applicable.

14. Feature selection and parsimony
   - After initial modeling and importance analysis, remove redundant or low-signal features to reduce complexity and improve explainability.

---

## Recommended preprocessing sequence (ordered steps)
1. Ingest raw data and run ingestion verification (MUST).
2. Clean categorical columns (normalize strings, correct typos) and check duplicates (MUST).
3. Confirm units/semantics for `int.rate` and `log.annual.inc` (MUST).
4. Create a baseline EDA notebook with distributions, skewness, missingness, and correlation matrix (MUST).
5. Split dataset into stratified train / validation / test sets. Keep test set untouched until final evaluation (MUST).
6. On training data only:
   - Apply outlier treatment (winsorize/log transform) and record transformation parameters (SHOULD).
   - Decide and apply encoding for categorical variables (SHOULD).
   - Fit scalers (Standard/Robust) on training set and apply to validation/test (SHOULD).
   - Implement resampling / class weighting within cross‑validation folds (SHOULD).
7. Train baseline models:
   - Logistic Regression (with regularization and class_weight) as benchmark.
   - Random Forest and Gradient Boosting as primary complex models.
   - SVM if dataset size and dimensionality allow.
8. Evaluate discrimination and calibration metrics:
   - ROC‑AUC, PR‑AUC, Precision/Recall at business thresholds.
   - Brier score and calibration curves for probability quality.
9. Calibrate final model probabilities if needed (Platt / isotonic) using a held‑out validation set or via CalibratedClassifierCV (SHOULD).
10. Produce model explainability outputs (feature importances, SHAP values if using tree/boosting models).

---

## Recommended evaluation metrics
- Discrimination: ROC‑AUC, PR‑AUC (PR especially important given class imbalance)
- Calibration: Brier score, calibration curve, expected calibration error
- Business metrics: Precision at specific recall thresholds, expected loss given predicted probability and exposure
- Stability: performance across time slices and subgroups (e.g., by `purpose`)

---

## Practical tooling and function references
(Use these within your code/pipeline where appropriate)
- Data splitting: sklearn.model_selection.train_test_split (use `stratify`)
- Encoders: sklearn.preprocessing.OneHotEncoder, OrdinalEncoder; category‑specific: category_encoders (target/frequency encoding), CatBoost native handling
- Scaling: sklearn.preprocessing.StandardScaler, RobustScaler
- Imbalance handling: sklearn.utils.class_weight, imblearn.over_sampling.SMOTE, XGBoost `scale_pos_weight`
- Outlier handling: pandas.quantile for caps, numpy/log transform
- Modelling: sklearn.linear_model.LogisticRegression, sklearn.ensemble.RandomForestClassifier, xgboost.XGBClassifier, lightgbm.LGBMClassifier, catboost.CatBoostClassifier, sklearn.svm.SVC
- Calibration: sklearn.calibration.CalibratedClassifierCV, sklearn.calibration.calibration_curve
- Explainability: SHAP, sklearn.inspection.permutation_importance

---

## Next steps / suggested short-term experiment plan
1. Produce an exploratory data analysis notebook that implements the MUST checklist, documents findings, and validates assumptions.
2. Implement a reproducible preprocessing pipeline (fit on training set; transform for validation/test).
3. Run baseline models (Logistic Regression, Random Forest, LightGBM/CatBoost) using stratified CV and evaluate discrimination + calibration.
4. Tune best performing model(s) and apply probability calibration. Produce calibration plots and business-oriented decision thresholds.
5. Document model selection rationale, feature importances, and example decision logic for deployment.

---

If you want, I can:
- Generate a ready-to-run preprocessing pipeline template (sklearn Pipeline + ColumnTransformer) that implements the MUST/SHOULD steps,
- Provide example notebooks for EDA, model training, calibration, and evaluation,
- Draft a model card (documentation) that captures performance and limitations for stakeholders.

Which of these would you like next?